# Color Diversity in Models (U-Net like with skip-connections)
## Notebook init

### Config

In [ ]:
CLEARML_SAVE_TASK = True

In [ ]:
CONFIG = {
    "batch_size": 32,
    "num_workers": 6,
    "seed": 42,
    "model_version": "v2",
}

### Libs

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
from pathlib import Path

import torch
from clearml import Dataset, Logger, Task, TaskTypes
from torch.utils.data import DataLoader

### Chromatica modules

In [ ]:
from chromatica.charts import color_diversity
from chromatica.datasets.dataset import ImageDataset
from chromatica.nn import load_cnn
from chromatica.metrics.color_diversity import (
    compute_delta_a_stats,
    compute_delta_a_weighted,
)

In [ ]:
CNN = load_cnn(CONFIG["model_version"])

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

### Useful functions

In [ ]:
def mps_autocast_or_null():
    if device.type == "mps":
        try:
            return torch.autocast("mps", dtype=torch.float16)
        except Exception:
            return nullcontext()
    return nullcontext()


class CachedPredictLoader:
    def __init__(
        self,
        model,
        base_loader,
        device,
        cache_dir: str | Path,
        *,
        overwrite: bool = False,
        prefetch_factor: int = 8,
        io_workers: int = 6,
    ):
        self.model = model
        self.base_loader = base_loader
        self.device = device
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.done_flag = self.cache_dir / "DONE"
        self.overwrite = overwrite
        self.prefetch_factor = max(1, int(prefetch_factor))
        self.io_workers = max(0, int(io_workers))

    def _write_mode(self) -> bool:
        return self.overwrite or not self.done_flag.exists()

    def __len__(self):
        return len(self.base_loader)

    def _flush_buffer(self, buffer: list[tuple[int, torch.Tensor]]):
        for idx, pred_cpu in buffer:
            tmp = self.cache_dir / f"{idx:06d}.pt.tmp"
            final = self.cache_dir / f"{idx:06d}.pt"
            torch.save({"pred": pred_cpu}, tmp)
            os.replace(tmp, final)

    def __iter__(self):
        if self._write_mode():
            was_training = self.model.training
            self.model.eval()

            ac = mps_autocast_or_null()
            buffer: list[tuple[int, torch.Tensor]] = []

            with torch.inference_mode():
                for i, (x, _, _) in enumerate(self.base_loader):
                    x = x.to(self.device, non_blocking=True)
                    with ac:
                        pred = self.model(x)
                    pred = pred.detach()

                    if self.device.type == "cuda":
                        pred_cpu = torch.empty_like(pred, device="cpu", pin_memory=True)
                        pred_cpu.copy_(pred, non_blocking=True)
                    else:
                        pred_cpu = pred.to("cpu")

                    yield (None, pred_cpu, None)

                    buffer.append((i, pred_cpu))
                    if len(buffer) >= self.prefetch_factor:
                        self._flush_buffer(buffer)
                        buffer.clear()

                if buffer:
                    self._flush_buffer(buffer)
                    buffer.clear()

            if was_training:
                self.model.train()

            self.done_flag.write_text("ok")

        else:
            files = sorted(self.cache_dir.glob("*.pt"))
            n = self.prefetch_factor

            for start in range(0, len(files), n):
                chunk = files[start : start + n]

                if self.io_workers > 0 and len(chunk) > 1:

                    def _load(path: Path):
                        return torch.load(path, map_location="cpu")["pred"]

                    with ThreadPoolExecutor(max_workers=self.io_workers) as ex:
                        loaded = list(ex.map(_load, chunk))
                else:
                    loaded = [torch.load(f, map_location="cpu")["pred"] for f in chunk]

                for pred_cpu in loaded:
                    yield (None, pred_cpu, None)

### ClearML init

In [ ]:
task = Task.init(
    project_name="Chromatica",
    task_name="Analyze diversity of colors in models",
    task_type=TaskTypes.testing,
    auto_connect_frameworks={"pytorch": False},
    tags=[
        CONFIG["model_version"],
    ],
)

In [ ]:
CONFIG = task.connect_configuration(CONFIG)

In [ ]:
train_task = Task.get_task(
    project_name="Chromatica",
    task_name="Train CNN for analyze diversity of colors",
    tags=[
        CONFIG["model_version"],
    ],
)

## Model based on `Food101`
### Prepare dataset

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

In [ ]:
dataset = ImageDataset(path / "test")

In [ ]:
model_food101_artifact = train_task.artifacts["model_food101"].get_local_copy()

In [ ]:
model_food101 = CNN()
model_food101.load_state_dict(torch.load(model_food101_artifact))
model_food101 = model_food101.to(device)

In [ ]:
loader = CachedPredictLoader(
    model_food101,
    DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        persistent_workers=True,
        shuffle=False,
        pin_memory=(device == torch.device("cuda")),
    ),
    device,
    cache_dir="./.data/color_diversity_in_models/food101/",
)

### Color Diversity
#### Color distribution histogram

In [ ]:
%%time

hist = color_diversity.compute_histogram(
    loader,
    nbins=250,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
)

In [ ]:
fig, _ = color_diversity.plot_histogram(hist, title="Food101 : Hue histogram")

Logger.current_logger().report_matplotlib_figure(
    title="Hue histogram",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=True,
)

#### Top Colors Pie Chart (Modes)

In [ ]:
%%time

top16_modes = color_diversity.compute_top_colors_modes(
    loader,
    topk=16,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
    input_scale=110.0,
    bin_size=5.0,
    merge_radius=5.0,
    chroma_floor=0.0,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_modes,
    l_value=50.0,
    title="Food101 : Top-16 Pie (Modes)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (Modes)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Palette Chart (Modes)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_modes,
    l_value=50.0,
    title="Food101 : Top-16 Palette (Modes)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (Modes)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Pie Chart (K-Means)

In [ ]:
top16_km = color_diversity.compute_top_colors_kmeans(
    loader,
    topk=16,
    input_scale=128.0,
    chroma_floor=8.0,
    sample_fraction=0.05,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
    init_centers_ab=top16_modes.centers_ab,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_km,
    l_value=50.0,
    title="Food101 : Top-16 Pie (K-Means)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (K-Means)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Palette (K-Means)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_km,
    l_value=50.0,
    title="Food101 : Top-16 Palette (K-Means)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (K-Means)",
    series="dataset=Food101",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Δa* Metrics (Food101)

In [ ]:
# Build an evaluation loader over GT to compute Δa* metrics
eval_loader = DataLoader(
    dataset,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    persistent_workers=True,
    shuffle=False,
    pin_memory=(device == torch.device("cuda")),
)

# Compute mean and median Δa* across pixels
delta_stats = compute_delta_a_stats(
    eval_loader,
    model_food101,
    device=device,
    gt_scale=110.0,
    pred_scale=110.0,
)
print("Food101 Δa* stats:", delta_stats)
Logger.current_logger().report_scalar(
    title="Δa* mean", series="dataset=Food101", value=delta_stats["mean"], iteration=0
)
Logger.current_logger().report_scalar(
    title="Δa* median",
    series="dataset=Food101",
    value=delta_stats["median"],
    iteration=0,
)

# Chroma-weighted Δa* (weights = C*_gt)
delta_weighted = compute_delta_a_weighted(
    eval_loader,
    model_food101,
    device=device,
    gt_scale=110.0,
    pred_scale=110.0,
    weight_source="gt",
)
print("Food101 Δa*_w (gt-weighted):", delta_weighted)
Logger.current_logger().report_scalar(
    title="Δa*_w (gt)", series="dataset=Food101", value=delta_weighted, iteration=0
)

## Model based on `COCO`
### Prepare dataset

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="COCO").get_local_copy()
)

In [ ]:
dataset = ImageDataset(path / "test")

In [ ]:
model_coco_artifact = train_task.artifacts["model_coco"].get_local_copy()

In [ ]:
model_coco = CNN()
model_coco.load_state_dict(torch.load(model_coco_artifact))
model_coco = model_coco.to(device)

In [ ]:
loader = CachedPredictLoader(
    model_coco,
    DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        persistent_workers=True,
        shuffle=False,
        pin_memory=(device == torch.device("cuda")),
    ),
    device,
    cache_dir="./.data/color_diversity_in_models/coco/",
)

### Color Diversity
#### Color distribution histogram

In [ ]:
%%time

hist = color_diversity.compute_histogram(
    loader,
    nbins=250,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
)

In [ ]:
fig, _ = color_diversity.plot_histogram(hist, title="COCO : Hue histogram")

Logger.current_logger().report_matplotlib_figure(
    title="COCO : Hue histogram",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Pie Chart (Modes)

In [ ]:
%%time

top16_modes = color_diversity.compute_top_colors_modes(
    loader,
    topk=16,
    sample_fraction=0.5,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
    input_scale=110.0,
    bin_size=5.0,
    merge_radius=5.0,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_modes,
    l_value=50.0,
    title="COCO : Top-16 Pie (Modes)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (Modes)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

#### Top Colors Palette Chart (Modes)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_modes,
    l_value=50.0,
    title="COCO : Top-16 Palette (Modes)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (Modes)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Pie Chart (K-Means)

In [ ]:
top16_km = color_diversity.compute_top_colors_kmeans(
    loader,
    topk=16,
    input_scale=128.0,
    chroma_floor=8.0,
    sample_fraction=0.05,
    rng=torch.Generator().manual_seed(CONFIG["seed"]),
    init_centers_ab=top16_modes.centers_ab,
)

In [ ]:
fig, _ = color_diversity.plot_top_colors_pie(
    top16_km,
    l_value=50.0,
    title="COCO : Top-16 Pie (K-Means)",
    include_other=False,
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Pie (K-Means)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Top Colors Palette (K-Means)

In [ ]:
fig, _ = color_diversity.plot_top_colors_palette(
    top16_km,
    l_value=50.0,
    title="COCO : Top-16 Palette (K-Means)",
    cols=8,
    tile_px=72,
    spacing_px=2,
    fmt="{:.1f}%",
)

Logger.current_logger().report_matplotlib_figure(
    title="Top-16 Palette (K-Means)",
    series="dataset=COCO",
    figure=fig,
    iteration=0,
    report_image=True,
    report_interactive=False,
)

### Δa* Metrics (COCO)

In [ ]:
# Build an evaluation loader over GT to compute Δa* metrics
eval_loader = DataLoader(
    dataset,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    persistent_workers=True,
    shuffle=False,
    pin_memory=(device == torch.device("cuda")),
)

# Compute mean and median Δa* across pixels
delta_stats = compute_delta_a_stats(
    eval_loader,
    model_coco,
    device=device,
    gt_scale=110.0,
    pred_scale=110.0,
)
print("COCO Δa* stats:", delta_stats)
Logger.current_logger().report_scalar(
    title="Δa* mean", series="dataset=COCO", value=delta_stats["mean"], iteration=0
)
Logger.current_logger().report_scalar(
    title="Δa* median", series="dataset=COCO", value=delta_stats["median"], iteration=0
)

# Chroma-weighted Δa* (weights = C*_gt)
delta_weighted = compute_delta_a_weighted(
    eval_loader,
    model_coco,
    device=device,
    gt_scale=110.0,
    pred_scale=110.0,
    weight_source="gt",
)
print("COCO Δa*_w (gt-weighted):", delta_weighted)
Logger.current_logger().report_scalar(
    title="Δa*_w (gt)", series="dataset=COCO", value=delta_weighted, iteration=0
)

In [ ]:
if CLEARML_SAVE_TASK:
    task.mark_completed()
else:
    task.close()
    task.set_archived(True)